# AI Character Studio — Train LoRA (Kaggle)
Kaggle GPU notebooks give ~29GB RAM (vs Colab free 12.7GB), enough to load FLUX without the system-RAM OOM.

**Before running:** enable **Internet** (Settings → Internet on), pick a **GPU** accelerator (T4 x2 or P100), add your `HF_TOKEN` under **Add-ons → Secrets**, and attach your raw images as a **Kaggle Dataset** (set `RAW_IMAGES_DIR` below).

In [ ]:
# Configuration
CHARACTER = "mayalin"
STEPS = 4000
REPO_URL = "https://github.com/maivandangkhoa/ai-character-studio.git"
# Your uploaded Kaggle Dataset of raw training images (20-40 imgs):
RAW_IMAGES_DIR = "/kaggle/input/mayalin-dataset"

import os
# HF token from Kaggle Secrets (Add-ons -> Secrets, name it HF_TOKEN). FLUX.1-dev is gated.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("Could not read HF_TOKEN secret — add it under Add-ons -> Secrets.", e)

In [ ]:
# Clone / update repo
import os, subprocess
REPO_DIR = "/kaggle/working/ai-character-studio"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Install dependencies
!python scripts/setup.py

In [ ]:
# Stage raw images from the attached Kaggle Dataset into the character's raw dir
import glob, shutil, os
from src.utils.paths import Paths

paths = Paths.load().ensure()
cp = paths.character(CHARACTER).ensure()
exts = (".png", ".jpg", ".jpeg", ".webp")
staged = 0
for src in glob.glob(os.path.join(RAW_IMAGES_DIR, "**", "*"), recursive=True):
    if src.lower().endswith(exts):
        dst = cp.raw / os.path.basename(src)
        if not dst.exists():
            shutil.copyfile(src, dst)
            staged += 1
print(f"Staged {staged} new images into {cp.raw}")

In [ ]:
# Download FLUX weights (cached on /kaggle/temp, off the 20GB persisted output)
!python scripts/download_models.py

In [ ]:
# Train (prepare -> caption -> train; auto-resumes from latest checkpoint)
!python scripts/train.py --character "$CHARACTER" --steps $STEPS

The exported LoRA bundle lands in `/kaggle/working/AICharacter/outputs/lora/<character>/`. **Save a version** (top-right → Save Version) to persist it, or download `lora.safetensors` directly from the output panel.